# Library

In [1]:
# Library cell
import os
import time
import numpy as np
import redpitaya_scpi as scpi
import mkf
import importlib
import socket_instrument
import pickle
from scipy.signal import get_window
importlib.reload(mkf)
importlib.reload(scpi)
import matplotlib.pyplot as plt
%matplotlib qt

import requests
def bring_up_scpi_server(host="rp-f0b916.local", stop=False):
    r = requests.get(f'http://{host}/get_scpi_status', auth=('user', 'pass'))
    assert r.status_code == 200, f"Error reaching {host}, status code: {r.status_code}"
    command, state1, state2 = ('stop', 'active', 'inactive') if stop else ('start', 'inactive', 'active')
    if r.text.strip('\n') == state1:
        r = requests.get(f'http://{host}/{command}_scpi_manager', auth=('user', 'pass'))
        assert r.status_code == 200, f"Fail starting scpi manager {host}, status code: {r.status_code}"
        r = requests.get(f'http://{host}/get_scpi_status', auth=('user', 'pass'))
        assert r.status_code == 200, f"Error reaching {host}, status code: {r.status_code}"
    if r.text.strip('\n') == state2:
        print(f"SCPI server {state2}")

def get_calibration_params(wv, figure_n=100):
    plt.figure(figure_n)
    plt.cla()
    ellipse_param = mkf.fit_ellipse(*wv)
    t = np.linspace(0,2*np.pi, 200)
    fitted_ellipse = mkf.rescale(np.sin(t), np.cos(t), ellipse_param, invert=True)
    x,y = mkf.rescale(*wv, ellipse_param)
    plt.scatter(*wv)
    plt.plot(*fitted_ellipse, color='red')
    y_inc = 1000/2**13
    print(f"CH1 DC = {ellipse_param[0]*y_inc:.0f} mV - min={min(wv[0])} max={max(wv[0])} DRU={100*max(wv[0])/2**13:.0f}%")
    print(f"CH2 DC = {ellipse_param[1]*y_inc:.0f} mV - min={min(wv[1])} max={max(wv[1])} DRU={100*max(wv[1])/2**13:.0f}%")
    print(f"Radius = {ellipse_param[3]*y_inc:.0f} mV")
    print(f"Eccentricity = {ellipse_param[2]:.2f}")
    print(f"Angle = {ellipse_param[4]*360/(2*np.pi):.1f}")
    return ellipse_param

def setup(ip, decimation, avg, delay, waveform_len, timeout=1, delimiter='\r\n', ch = [1,2]):
    n_ch = len(ch)
    dig = scpi.scpi(ip, timeout=timeout) 
    dig.delay = delay
    dig.waveform_len = waveform_len
    dig.tx_txt('ACQ:RST')
    start_address = int(dig.txrx_txt('ACQ:AXI:START?'))
    size = int(dig.txrx_txt('ACQ:AXI:SIZE?'))
    print(f'Memory reserved for acquisition {size/1E6:.2f} MB')
    start_address2 = round(start_address + size/2) if n_ch == 2 else start_address
    dig.tx_txt(f"ACQ:AXI:DEC {decimation}")
    dig.tx_txt(f'ACQ:AVG {avg}')
    dig.tx_txt('ACQ:AXI:DATA:UNITS RAW')
    dig.tx_txt('ACQ:DATA:FORMAT BIN')
    dig.tx_txt(f"ACQ:AXI:SOUR1:Trig:Dly {waveform_len + 1 - delay}")
    dig.tx_txt(f"ACQ:AXI:SOUR2:Trig:Dly {waveform_len + 1 - delay}")
    if 1 in ch:
        dig.tx_txt(f"ACQ:AXI:SOUR1:SET:Buffer {start_address},{size/n_ch}")
        dig.tx_txt('ACQ:AXI:SOUR1:ENable ON')
    if 2 in ch:
        dig.tx_txt(f"ACQ:AXI:SOUR2:SET:Buffer {start_address2},{size/n_ch}")
        dig.tx_txt('ACQ:AXI:SOUR2:ENable ON')    
    return dig

def wait_acquisition(dig):
    j = 0
    while 1:
        j += 1
        if j > 10:
            print("Took to more than 100 ms to trigger!")
            break
        dig.tx_txt("ACQ:TRig:STAT?")
        if dig.rx_txt() == 'TD':
            print("Triggered", end="\r")
        break
        time.sleep(.01) 
        print(f"Waiting trigger:{j}               ", end="\r")
    
    # wait for fill adc buffer
    j = 0
    while 1:
        j += 1
        dig.tx_txt('ACQ:AXI:SOUR1:TRIG:FILL?')
        if dig.rx_txt() == '1':
            print(f'DMA buffer full           ', end="\r")
            break
        time.sleep(.01)
        print(f"Waiting acquisition:{j}               ", end="\r")
        if j >20/0.01:
            print("Verificar Conexão WIfi ou Rede", end="\r")


def get_data(dig, waveforms, READ_waveform_len=2**10, chunk_size=2**16, ch=[1,2]):
    ## Get write pointer at trigger location
    posChA, posChB = 0,0
    if 1 in ch:
        posChA = int(dig.txrx_txt('ACQ:AXI:SOUR1:Trig:Pos?'))
    if 2 in ch:
        posChB = int(dig.txrx_txt('ACQ:AXI:SOUR2:Trig:Pos?'))
    #print(f'posChA={posChA:_} posChB={posChB:_}')
    N = READ_waveform_len//chunk_size
    remainder = READ_waveform_len%chunk_size
    for i in range(N):
        print(f"{i}/{N}", end='\r')
        if 1 in ch:
            dig.tx_txt(f"ACQ:AXI:SOUR1:DATA:Start:N? {posChA+i*chunk_size - dig.delay},{chunk_size}")
            buf1 = dig.rx_arb()
            waveforms[0][i*chunk_size:(i+1)*chunk_size] = np.frombuffer(buf1, dtype=np.dtype('>i2'))
        if 2 in ch:
            dig.tx_txt(f"ACQ:AXI:SOUR2:DATA:Start:N? {posChB+i*chunk_size - dig.delay},{chunk_size}")
            buf2 = dig.rx_arb()
            waveforms[1][i*chunk_size:(i+1)*chunk_size] = np.frombuffer(buf2, dtype=np.dtype('>i2'))
    if remainder:
        if 1 in ch:
            dig.tx_txt(f"ACQ:AXI:SOUR1:DATA:Start:N? {posChA+N*chunk_size - dig.delay},{remainder}")
            buf1 = dig.rx_arb()
            waveforms[0][N*chunk_size:N*chunk_size+remainder] = np.frombuffer(buf1, dtype=np.dtype('>i2'))
        if 2 in ch:
            dig.tx_txt(f"ACQ:AXI:SOUR2:DATA:Start:N? {posChB+N*chunk_size - dig.delay},{remainder}")
            buf2 = dig.rx_arb()
            waveforms[1][N*chunk_size:N*chunk_size+remainder] = np.frombuffer(buf2, dtype=np.dtype('>i2'))       


def get_data(dig, waveforms, READ_waveform_len=2**10, chunk_size=2**16, ch=[1,2]):
    READ_waveform_len = waveforms.shape[1]
    ## Get write pointer at trigger location
    posChA, posChB = 0,0
    if 1 in ch:
        posChA = int(dig.txrx_txt('ACQ:AXI:SOUR1:Trig:Pos?'))
    if 2 in ch:
        posChB = int(dig.txrx_txt('ACQ:AXI:SOUR2:Trig:Pos?'))
    #print(f'posChA={posChA:_} posChB={posChB:_}')
    N = READ_waveform_len//chunk_size
    remainder = READ_waveform_len%chunk_size
    for i in range(N+1) if remainder else range(N):
        #print(f"{i}/{N}", end='')
        i_start = i*chunk_size
        if i < N:
            n_points = chunk_size
        else:
            n_points = remainder
        for c in ch:
            i_end = i_start + n_points
            dig.tx_txt(f"ACQ:AXI:SOUR{c}:DATA:Start:N? {posChA + i_start - dig.delay},{n_points}")
            buf = dig.rx_arb()
            waveforms[c-1][i_start:i_end] = np.frombuffer(buf, dtype=np.dtype('>i2'))
 

def tear_down(dig):
    ## Close connection with Red Pitaya
    print('Releasing resources')
    dig.tx_txt('ACQ:STOP')
    dig.tx_txt('ACQ:AXI:SOUR1:ENable OFF')
    dig.tx_txt('ACQ:AXI:SOUR2:ENable OFF')
    dig.close()

def get_calibration_params(wv, figure_n=100):
    if len(wv.shape) == 3:
        wv = wv[:,:,0].T
    plt.figure(figure_n)
    plt.cla()
    ellipse_param = mkf.fit_ellipse(*wv)
    t = np.linspace(0,2*np.pi, 200)
    fitted_ellipse = mkf.rescale(np.sin(t), np.cos(t), ellipse_param, invert=True)
    x,y = mkf.rescale(*wv, ellipse_param)
    plt.scatter(*wv)
    plt.plot(*fitted_ellipse, color='red')
    y_inc = 1000/2**13
    print(f"CH1 DC = {ellipse_param[0]*y_inc:.0f} mV - min={min(wv[0])} max={max(wv[0])} DRU={100*max(wv[0])/2**13:.0f}%")
    print(f"CH2 DC = {ellipse_param[1]*y_inc:.0f} mV - min={min(wv[1])} max={max(wv[1])} DRU={100*max(wv[1])/2**13:.0f}%")
    print(f"Radius = {ellipse_param[3]*y_inc:.0f} mV")
    print(f"Eccentricity = {ellipse_param[2]:.2f}")
    print(f"Angle = {ellipse_param[4]*360/(2*np.pi):.1f}")
    return ellipse_param


def demodulate(waveforms, param = None):
    demo = lambda x, param: np.unwrap(np.arctan2(*mkf.rescale(*x, param)))
    shape = waveforms.shape
    if len(shape) == 2:
        if ellipse_param is None:
            raise Exception("Needs ellipse_param")
        if shape[0] == 2:
            return demo(data.waveforms, param)
        elif shape[1] == 2:
            return demo(data.waveforms.T, param)
        else:
            raise Exception("One of the dimension needs to be 2")
    if len(shape) == 3:
        demodulated = np.empty((shape[0], shape[2]))
        for i in range(shape[0]):
            print(f"Demodulating={i+1:04d}/{shape[0]}", end='\r')
            demodulated[i] =  demo(data.waveforms[i], param)
        return demodulated

def process_sweep_frequency():
    global data, n_freqs, n_ch, waveform_size, t, ellipse_params, demodulated, N, window, freqs, amp, angles, std, avg, filename, subdiv
    n_freqs, n_ch, waveform_size = data.waveforms.shape   
    N = waveform_size//subdiv
    window = get_window('blackman', N)
    freqs = data.excitation_frequencies
    amp = np.empty((subdiv, n_freqs))
    angles = np.empty((subdiv, n_freqs))
    for i in range(n_freqs):
        t = np.arange(waveform_size)*data.decimations[i]/data.base_sample_frequency
        print(f"{i+1:04d}/{n_freqs}", end='\r')
        for j in range(subdiv):
            segment = demodulated[i][N*j:N*(j+1)]
            complex_amp = sum(segment*window*np.exp(2j*np.pi*freqs[i]*t[:N]))*2/N
            amp[j, i] = np.abs(complex_amp)
            angles[j, i] = np.angle(complex_amp)
    std = amp.std(axis=0)
    avg = np.average(amp, axis=0)


def process_sweep_frequency():
    global data, n_freqs, n_ch, waveform_size, t, ellipse_params, demodulated, N, window, freqs, amp, angles, std, avg, filename, subdiv
    n_freqs, n_ch, waveform_size = data.waveforms.shape   
    N = waveform_size//subdiv
    window = filter.get_window('blackman', N)
    freqs = data.excitation_frequencies
    amp = np.empty((subdiv, n_freqs))
    angles = np.empty((subdiv, n_freqs))
    for i in range(n_freqs):
        t = np.arange(waveform_size)*data.decimations[i]/data.base_sample_frequency
        print(f"{i+1:04d}/{n_freqs}", end='\r')
        for j in range(subdiv):
            segment = demodulated[i][N*j:N*(j+1)]
            complex_amp = sum(segment*window*np.exp(2j*np.pi*freqs[i]*t[:N]))*2/N
            amp[j, i] = np.abs(complex_amp)
            angles[j, i] = np.angle(complex_amp)
    std = amp.std(axis=0)
    avg = np.average(amp, axis=0)
    
import winsound
import time

def beep_start():
     # Frequência e duração do som de início
     winsound.Beep(1000, 1500)  # 1000 Hz, 500 ms

def beep_stop():
     # Frequência e duração do som de parada
     winsound.Beep(500, 1500)  # 500 Hz, 500 ms



# Aquisição sem gerador de sinal nem trigger

In [1]:
time.date

NameError: name 'time' is not defined

Red Pitaya Official Documentation
- [SCPI](https://redpitaya.readthedocs.io/en/latest/appsFeatures/remoteControl/command_list.html)
- [Deep Memory Acquisition (DMA)](https://redpitaya.readthedocs.io/en/latest/appsFeatures/examples/DMA/deepMemoryAcq.html)
- [Red Pitay Forum](https://forum.redpitaya.com/)
- [Setting reserved memory for DMA (default is 2MB)](https://redpitaya.readthedocs.io/en/latest/appsFeatures/remoteControl/deepMemoryAcquisition.html#changing-reserved-memory)

### How to run
- Execute the **Library** cell.
- Choose variables below.
- **waveform_len** : Sets number of points in the waveform, max 50 millions. Files size is 4 times this value in bytes. Acquisition duration in seconds is waveform_len*decimation/125E6.
- **decimation** : Average or skips number of points equal to **decimation** value. To average set **avg** to **'ON'**. To skip set **avg** to **'OFF'**.
- Run cell to calculate básic parameters. 

In [4]:
waveform_len = int(2**25)# Sets number of points in the waveform, max 50 millions. Files size is 4 times this value in bytes. 
decimation = 2**1

avg = 'OFF'
delay = 0000 #Delay from first point in the waveform to trigger

##############################################
### don't change variables below
#######################################
sample_frequency = 125E6
sample_frequency_eff = sample_frequency/decimation
acquisistion_time = waveform_len/sample_frequency_eff
print(f"Acquistion time: {acquisistion_time*1000:.2f} ms\nEffective sample frequency: {sample_frequency_eff:_.0f} SPS\nData size : {4*waveform_len/1E6:.2f} MB")
print(f"Number of points: {waveform_len:_d}")
if avg=='ON':
    print(f'Average: {decimation} points')
else:
    print(f'Decimation: {decimation} points')

Acquistion time: 536.87 ms
Effective sample frequency: 62_500_000 SPS
Data size : 134.22 MB
Number of points: 33_554_432
Decimation: 2 points


In [282]:
time.sleep(3)
beep_start()  # Apita para abrir valvula
time.sleep(1)
bring_up_scpi_server(host="rp-f0b916.local")

bring_up_scpi_server(host="rp-f0b916.local", stop=False)
dig = setup('rp-f0b916.local', decimation, avg, delay, waveform_len, ch=[1,2])
print(f'Acquiring', end="\r")
dig.tx_txt('ACQ:START')
dig.tx_txt('ACQ:TRig NOW')
wait_acquisition(dig)
waveforms = np.zeros((2, waveform_len), dtype=np.dtype('>i2'))
get_data(dig, waveforms, waveform_len, chunk_size = 2**18)
tear_down(dig)
#plt.cla()
#plt.plot(waveforms.T)
#beep_stop()  # Apita para fechar valvula

SCPI server active
SCPI server active
Connected to REDPITAYA,INSTR2024,0,01-16
Memory reserved for acquisition 201.33 MB
Releasing resources                 


In [283]:
waveforms.shape

(2, 33554432)

In [285]:
data = mkf.dotdict()



t = np.arange(waveform_len)/sample_frequency_eff
data.sample_frequency_effective = sample_frequency_eff
data.sample_frequency = sample_frequency
data.decimation = decimation
data.average_state = avg
data.delay = delay
data.waveforms = waveforms
data.descricao = "Medida_Ruído_do_Laser"
data.voltage = 1.
data.voltage_unit = 'V'

data.position = 2.
data.position_unit = 'cm'
data.ruido= False
data.flow_unit = 'L-min'
data.preassure = 10.0
data.flow = 10.0
data.preassure_unit = 'Bar'
#data.valve_state = 'closed'
data.valve_state = 'open'
data.sensor = 'polyurethane'

#Medida Ruído do laser

data.OPD = 280.
data.Laser_PWR = 5.
data.Tap = 'Sim'
#data.Tap = 'Não_3'



#data.sensor = 'nylon'
#data.sensor = 'steel'

In [286]:
filename = f"{data.descricao} Laser_PWR-{data.Laser_PWR:.1f}dBm_OPD-{data.OPD:.1f}mm_Vibração-{data.Tap}"
print(filename)

Medida_Ruído_do_Laser Laser_PWR-5.0dBm_OPD-280.0mm_Vibração-Sim


In [287]:
import pickle
directory = "Raw Data/Medidas de Ruído de Fase - Laser/"
pickle.dump(dict(data), open(directory+filename+'.pickle', 'wb'))

# Aquisição de sinal com uma frequencia fixa
Tem que configurar as configurações no gerador de função. Aqui ele só confere se o gerador de função está ligado

In [ ]:
##############################
### Início configuração
##############################
waveform_len = int(2**12)# Sets number of points in the waveform, max 50 millions. Files size is 4 times this value in bytes. 
decimation = 2**6
avg = 'OFF'
delay = 0000 #Delay from first point in the waveform to trigger

IP_gerador_de_onda = "192.168.1.194"
IP_red_pitaya = 'rp-f0b916.local'

#############################
### Fim configuração
###############################

bring_up_scpi_server(host=IP_red_pitaya)
sg = scpi.scpi(IP_gerador_de_onda, port=5555,  timeout=1, delimiter='\n') 
N = 2**14
dig = setup(ip='rp-f0b916.local', decimation=2**14, avg="OFF", delay=0, waveform_len=waveform_len)
if (sg.txrx_txt("OUTP1?") != "ON"):
    print("\x1b[31m\"rSignal generator off!\x1b[0m ")
dig.tx_txt('ACQ:START')
dig.tx_txt('ACQ:TRig NOW') 
wait_acquisition(dig)
waveform = np.zeros((2,waveform_len))
get_data(dig, waveform)
tear_down(dig)
sg.close()
_ = get_calibration_params(waveform)
# check if there is signal
plt.figure(101)
plt.cla()
plt.plot(waveform[0])
plt.plot(waveform[1])

In [ ]:
plt.plot(wv[0])

# Aquisição de sinal varrendo frequencia com trigger
Adquire uma forma de onda (waveform) para cada frequencia diferente usando o gerador de função.
## Como usar
- É interessante usar a célula "Aquisição de sinal com uma frequencia fixa" para verificar se tudo está funcionando bem. A aquisição - pode demorar vários minutos.
- Configure as opções abaixo e execute a célula.
- Se a aquisição estiver satisfatória. Salve os dados usando a celula #salvar dados

 # Aquisição em Baixa Frequência:

In [39]:

#freespace = True
freespace = False


In [41]:
#External Trigger Positive Edge

#####################################
### Início da configuração
#####################################




        
waveform_len = 2**18
delay = 0.1 # tempo de espera entre a mudança de frquencia e o inicio da medida
decimation = 2**10
avg = 'ON'   
freqs = np.logspace(np.log10(100),np.log10(1_000), 100)

    
#freqs = np.arange(10000, 1000000, 1000)
IP_gerador_de_onda = "192.168.1.194"
IP_red_pitaya = 'rp-f0b916.local'

#####################################
### Fim do trecho de configuração
#####################################
sample_frequency = 125E6 # esse parametro é da red pitaya. Se quiser mexer nisso altere o decimation
print(f'Taxa de amostragem efetiva = {sample_frequency/decimation:_}')
bring_up_scpi_server(host="rp-f0b916.local")
data = mkf.dotdict()
sg = scpi.scpi(IP_gerador_de_onda, port=5555,  timeout=1, delimiter='\n') 
dig = setup(ip=IP_red_pitaya, decimation=decimation, avg=avg, delay=0, waveform_len=waveform_len)
n_freqs = len(freqs)
waveforms = np.empty((n_freqs, 2, waveform_len), dtype=np.dtype('>i2'))
low_dec_not_set = False
for i, freq in enumerate(freqs):
    sg.tx_txt(f"SOUR1:FREQ {freqs[i]:.0f}")
    time.sleep(delay)
    dig.tx_txt('ACQ:START')
    dig.tx_txt('ACQ:TRig EXT_PE')   
    wait_acquisition(dig)
    print(f'Acquiring {i:04d}/{n_freqs:04d}', end="\r")
    get_data(dig, waveforms[i])
tear_down(dig)
sg.close()
#t = np.arange(waveform_len)*dec/sample_frequency

Taxa de amostragem efetiva = 122_070.3125
SCPI server active
Connected to Rigol Technologies,DG1022Z,DG1ZA260300077,03.01.12  
Connected to REDPITAYA,INSTR2024,0,01-16
Memory reserved for acquisition 201.33 MB
Releasing resources                  


In [42]:
################################################################
#### Salvar os dados
####
#### Escrever os detalhes da medidas que achar necessário
#################################################################
data.sample_frequency_effective = sample_frequency/decimation
data.sample_frequency = sample_frequency
data.decimation = decimation
data.average_state = avg
data.delay = delay
data.waveforms = waveforms
data.excitation_frequencies = freqs
data.comentarios = 'sensor com duas bobinas de 15m embotidas em epoxy de alta temperatura dendro de cilindro de inox.'
data.material = 'Epoxy Embutido - INOX'
data.temperatura = '24°C'
data.piezo = 'frequency_piezo_16000x1240'
data.comprimento_da_fibra = '15'
data.comprimento_da_fibra_unit = 'm'
data.date = time.localtime()
data.signal_generator_amplitude = '16'
data.signal_generator_amplitude_unit = 'V'
if freespace:
    data.tipo_de_medida = 'freespace'
else: 
    data.tipo_de_medida = 'sensor'
directory = "Medidas espectro"
filename = data.piezo + '_' + data.material +'_'+ data.temperatura + '_coil_'+ data.comprimento_da_fibra + data.comprimento_da_fibra_unit + '_' + data.tipo_de_medida 

filename = filename +'_low_freq'



print(filename)

#######################################################
#### fim da configuração se salvar os dados
#######################################################
import os
import pickle
if not filename.endswith('.pickle'):
    filename += '.pickle'
data_dir = "Raw Data"
assert os.path.exists(data_dir), "O notebook não está no diretorio certo, navegue usando cd até o diretorio certo. Use pwd para saber em que diretorio que você está."
file_full_path = os.path.join(os.getcwd(),data_dir, directory, filename)
data.data_file_path = file_full_path
save = False
if os.path.exists(file_full_path):
    if input("Arquivo já existe sobrescrever [s/N]:") == 's':
        save = True
else:
    save = True
if save:
    pickle.dump(dict(data), open(file_full_path, 'wb'))
    print("Arquivo salvo.")
else:
    print("Arquivo NÃO foi salvo.")

frequency_piezo_16000x1240_Epoxy Embutido - INOX_24°C_coil_15m_sensor_low_freq


Arquivo já existe sobrescrever [s/N]: s


Arquivo salvo.


 # Aquisição em Alta Frequência:

In [45]:
freespace

False

In [47]:
#External Trigger Positive Edge

#####################################
### Início da configuração
#####################################



waveform_len = 2**18
delay = 0.1 # tempo de espera entre a mudança de frquencia e o inicio da medida
decimation = 2**6
avg = 'ON'   
freqs = np.logspace(np.log10(1000),np.log10(1_000_000), 1000)

    
#freqs = np.arange(10000, 1000000, 1000)
IP_gerador_de_onda = "192.168.1.194"
IP_red_pitaya = 'rp-f0b916.local'

#####################################
### Fim do trecho de configuração
#####################################
sample_frequency = 125E6 # esse parametro é da red pitaya. Se quiser mexer nisso altere o decimation
print(f'Taxa de amostragem efetiva = {sample_frequency/decimation:_}')
bring_up_scpi_server(host="rp-f0b916.local")
data = mkf.dotdict()
sg = scpi.scpi(IP_gerador_de_onda, port=5555,  timeout=1, delimiter='\n') 
dig = setup(ip=IP_red_pitaya, decimation=decimation, avg=avg, delay=0, waveform_len=waveform_len)
n_freqs = len(freqs)
waveforms = np.empty((n_freqs, 2, waveform_len), dtype=np.dtype('>i2'))
low_dec_not_set = False
for i, freq in enumerate(freqs):
    sg.tx_txt(f"SOUR1:FREQ {freqs[i]:.0f}")
    time.sleep(delay)
    dig.tx_txt('ACQ:START')
    dig.tx_txt('ACQ:TRig EXT_PE')   
    wait_acquisition(dig)
    print(f'Acquiring {i:04d}/{n_freqs:04d}', end="\r")
    get_data(dig, waveforms[i])
tear_down(dig)
sg.close()
#t = np.arange(waveform_len)*dec/sample_frequency

Taxa de amostragem efetiva = 1_953_125.0
SCPI server active
Connected to Rigol Technologies,DG1022Z,DG1ZA260300077,03.01.12  
Connected to REDPITAYA,INSTR2024,0,01-16
Memory reserved for acquisition 201.33 MB
Releasing resources                 


In [53]:
################################################################
#### Salvar os dados
####
#### Detalhes das meddais na celula anterior
#################################################################

data.sample_frequency_effective = sample_frequency/decimation
data.sample_frequency = sample_frequency
data.decimation = decimation
data.average_state = avg
data.delay = delay
data.waveforms = waveforms
data.excitation_frequencies = freqs
data.comentarios = 'sensor com duas bobinas de 15m embotidas em epoxy de alta temperatura dendro de cilindro de inox.'
data.material = 'Epoxy Embutido - INOX'
data.temperatura = '24°C'
data.piezo = 'frequency_piezo_16000x1240'
data.comprimento_da_fibra = '15'
data.comprimento_da_fibra_unit = 'm'
data.date = time.localtime()
data.signal_generator_amplitude = '16'
data.signal_generator_amplitude_unit = 'V'
if freespace:
    data.tipo_de_medida = 'freespace'
else: 
    data.tipo_de_medida = 'sensor'

directory = "Medidas espectro"
filename = data.piezo + '_' + data.material +'_'+ data.temperatura + '_coil_'+ data.comprimento_da_fibra + data.comprimento_da_fibra_unit + '_' + data.tipo_de_medida 
filename = filename +'_high_freq'
  

    
#######################################################
#### fim da configuração se salvar os dados
#######################################################
import os
import pickle
if not filename.endswith('.pickle'):
    filename += '.pickle'
data_dir = "Raw Data"
assert os.path.exists(data_dir), "O notebook não está no diretorio certo, navegue usando cd até o diretorio certo. Use pwd para saber em que diretorio que você está."
file_full_path = os.path.join(os.getcwd(),data_dir, directory, filename)
data.data_file_path = file_full_path
save = False
if os.path.exists(file_full_path):
    if input("Arquivo já existe sobrescrever [s/N]:") == 's':
        save = True
else:
    save = True
if save:
    pickle.dump(dict(data), open(file_full_path, 'wb'))
    print("Arquivo salvo.")
else:
    print("Arquivo NÃO foi salvo.")
print(filename)

Arquivo já existe sobrescrever [s/N]: s


Arquivo salvo.
frequency_piezo_16000x1240_Epoxy Embutido - INOX_24°C_coil_15m_sensor_high_freq.pickle


In [ ]:
plt.plot(waveforms[0].T)

In [ ]:
#External Trigger Positive Edge - maybe external triggering might not be a good idea
reset_delay = 2
off_freq = 50_000
on_freq = 115_000
waveform_len = 2**14
dec = 2**3
avg = 'ON'
sample_frequency = 125E6


bring_up_scpi_server(host="rp-f0b916.local")
data = mkf.dotdict()
sg = scpi.scpi("192.168.1.194", port=5555,  timeout=1, delimiter='\n') 
dig = setup(ip='rp-f0b916.local', dec=dec, avg=avg, delay=0, waveform_len=waveform_len)


delays = np.logspace(np.log10(0.05), np.log10(2), 50 )
waveforms = np.empty((len(delays), 2, waveform_len), dtype=np.dtype('>i2'))

for i, delay in enumerate(delays):
    sg.tx_txt(f"SOUR1:FREQ {off_freq:.0f}")
    time.sleep(reset_delay)
    sg.tx_txt(f"SOUR1:FREQ {on_freq:.0f}")
    time.sleep(delay)
    print(f'Acquiring {i}', end="\r")
    dig.tx_txt('ACQ:START')
    dig.tx_txt('ACQ:TRig EXT_PE')   
    wait_acquisition(dig)
    get_data(dig, waveforms[i])
    
tear_down(dig)
sg.close()
t = np.arange(waveform_len)*dec/sample_frequency
data.sample_frequency_effective = sample_frequency/dec
data.sample_frequency = sample_frequency
data.decimation = dec
data.average_state = avg
data.on_frequency = on_freq
data.off_frequency = off_freq
data.reset_delay = reset_delay
data.delays = delays
data.waveforms = waveforms

In [63]:
import pickle
with open('medidas pulso/delay_sweep_test.pickle', 'wb') as fd:
    pickle.dump(data, fd)

In [4]:
import os

In [4]:
os.getcwd()

'/media/bernardo/onedrive/Empresa 2/Lab'

In [9]:
os.path.join(os.getcwd(),"Raw Data", 'f')

'/media/bernardo/onedrive/Empresa 2/Lab/Raw Data/f'

In [3]:
os.path.exists("Raw Data")

True

Arq dfd


In [6]:
a

'dfd'